# Treinar o detector de EPI (`epi_yolo.pt`) no Google Colab

**Trabalho Final — Detecção de Uso de EPI** · Equipe Ctrl+C, Ctrl+V e Fé · ESZA019 2026.2

Este notebook gera o arquivo **`epi_yolo.pt`** usado pelo `deteccao_epi.py`. Ele roda no **Google Colab**,
que oferece **GPU gratuita** — não é preciso ter placa de vídeo em casa.

**Como usar (para leigos):**
1. Abra https://colab.research.google.com , faça login com uma conta Google e envie (*Upload*) este arquivo `.ipynb`.
2. No menu **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**.
3. Execute as células **em ordem**, de cima para baixo (botão ▶ em cada uma).
4. No fim, o `epi_yolo.pt` será baixado para o seu computador. Coloque-o na pasta `Trabalho Final/codigos/`.

Tempo estimado: 20–60 min. Não precisa marcar imagens à mão — usaremos um dataset público já anotado.


## 1. Verificar a GPU
Se aparecer uma tabela com "Tesla T4" (ou similar), a GPU está ativa. Se der erro, volte ao passo 2 acima.

In [ ]:
!nvidia-smi

## 2. Instalar as bibliotecas
Instala o YOLO (Ultralytics) e o cliente do Roboflow (para baixar o dataset).

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. Baixar o dataset de EPI (já anotado)

Vamos usar o dataset público **Construction Site Safety** (Roboflow Universe), que já vem anotado em
YOLOv8 e tem as classes que precisamos — **Hardhat** (capacete), **Safety Vest** (colete) e **Person**
(pessoa). Você **não precisa marcar nenhuma imagem**.

Passos:
1. Crie uma conta gratuita em https://roboflow.com .
2. Abra o dataset: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety
3. Clique em **Download Dataset → Format: YOLOv8 → "show download code"**.
4. Copie o trecho mostrado (ele já vem com a **sua API key** e o **número da versão** corretos) e **cole
   no lugar da célula abaixo**. Ele será parecido com o exemplo — troque `SUA_API_KEY` e confira o número
   em `.version(...)` pelo que o Roboflow indicar.

> As classes `NO-Hardhat`, `NO-Safety Vest`, `Mask`, `Safety Cone`, etc. são simplesmente ignoradas pelo
> nosso `deteccao_epi.py`; usamos apenas Hardhat, Safety Vest e Person.

In [ ]:
# Cole aqui o trecho que o Roboflow mostra em "show download code" (Format: YOLOv8).
# Exemplo (troque a API key; confira o numero da versao no proprio Roboflow):
from roboflow import Roboflow
rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(27)          # <-- use o numero de versao que o Roboflow indicar
dataset = version.download("yolov8")

print("Dataset baixado em:", dataset.location)

**Alternativa sem Roboflow (Kaggle):** o mesmo dataset está espelhado em
https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow .
Baixe o `.zip`, envie no Colab (painel de arquivos à esquerda) e descompacte; depois aponte o
`data=` da célula de treino para o `data.yaml` que vem dentro dele. (O caminho do Roboflow, acima, é o
mais simples.)

## 4. Treinar o modelo
Parte de um YOLOv8 pré-treinado (*transfer learning*) e ajusta para as classes de EPI. Comece com
`epochs=50`; se o resultado for fraco, aumente para 100.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolov8n.pt")          # modelo leve, bom para rodar depois em CPU
modelo.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
)

## 5. Avaliar (mAP, precisão, revocação)
Números para colocar no Relatório Técnico (item 4 do manual).

In [ ]:
metricas = modelo.val()
print("mAP@0.5      :", metricas.box.map50)
print("mAP@0.5:0.95 :", metricas.box.map)

## 6. Baixar o `epi_yolo.pt`
Copia o melhor modelo treinado e baixa para o seu computador. Depois, coloque-o em
`Trabalho Final/codigos/` e rode `python3 deteccao_epi.py --modelo epi_yolo.pt`.

In [ ]:
import shutil, glob
melhor = sorted(glob.glob("runs/detect/train*/weights/best.pt"))[-1]
shutil.copy(melhor, "epi_yolo.pt")
print("gerado epi_yolo.pt a partir de", melhor)

from google.colab import files
files.download("epi_yolo.pt")

## Pronto!
Agora, na máquina do laboratório:
```bash
conda activate CV26
cd .../Trabalho\ Final/codigos
python3 deteccao_epi.py --modelo epi_yolo.pt --camera 0 --calib calib_camera.xml
```
Coloquem/tirem capacete e colete diante da câmera para ver **CONFORME / NÃO CONFORME**.
